# 16S Analyses of the Longitudinal Acne Study
## rCLR transformed top features

Date last updated: 1/2/2026

Notebook author: Britta De Pessemier, Yang Chen

Data analysis by: Tyler Myers, Britta De Pessemier, and Yang Chen

In [258]:
# Import essential Python packages
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Ellipse
from matplotlib.colors import LinearSegmentedColormap

# Scientific and statistical packages
import scipy.stats as ss
from skbio import DistanceMatrix
from skbio.stats.ordination import OrdinationResults
from skbio.stats.distance import permanova
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# BIOM and Qiime2 related packages
import biom
import qiime2 as q2
from biom import load_table
from qiime2 import Artifact
import h5py

# Gemelli RPCA and rarefaction utilities
import gemelli
from gemelli.preprocessing import rclr_transformation

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    message="You passed a edgecolor/edgecolors .*"
)

In [259]:
# Helper function to calculate prevalence
def calculate_prevalence(biom_table):
    # Convert the biom table's data to a dense array and calculate feature prevalence
    feature_presence = (biom_table.matrix_data > 0).astype(int)
    prevalence = feature_presence.sum(axis=1).A1 / biom_table.shape[1]  # Convert to 1D array using .A1
    return pd.Series(prevalence, index=biom_table.ids(axis='observation'))

In [260]:
# user paths and settings
TAXONOMY_TSV = "../Taxonomy/174116_taxonomy.tsv"
METADATA_TSV = "../Metadata/metadata_final_22102024.tsv"
CTF_QZA_DIR  = "../Analyses/16S/CTF/subject_biplot_qza"
BIOM_DIR     = "../Data/16S/Tables"
OUTDIR       = "../Figures/Supplementary/Suppl_Figure_2"

os.makedirs(OUTDIR, exist_ok=True)

In [261]:
group_colors = {
    "Healthy": "#423fa6",
    "Acne_NL": "#67b2be",
    "Acne_L":  "#df7963"
}

GROUP_ORDER = ["Healthy", "Acne_NL", "Acne_L"]
regions = ["V1-V3", "V4"]
cheek_sides = ["left", "right"]

def extract_last_taxon(taxon):
    if pd.isna(taxon):
        return "Unassigned"
    parts = [p.strip() for p in taxon.split(";")]
    if "uncultured" in taxon and len(parts) >= 3:
        return parts[-3]
    return parts[-1]

def add_significance_bracket(ax, x1, x2, y, text):
    ax.plot([x1, x2], [y, y], lw=1.2, c="black")
    ax.text((x1 + x2) / 2, y, text, ha="center", va="bottom", fontsize=10)

def extract_last_taxon(taxon):
    if pd.isna(taxon):
        return "Unassigned"
    parts = [p.strip() for p in taxon.split(";")]
    if "uncultured" in taxon and len(parts) >= 3:
        return parts[-3]
    return parts[-1]

def add_significance_bracket(ax, x1, x2, y, text):
    ax.plot([x1, x2], [y, y], lw=1.2, c="black")
    ax.text((x1 + x2) / 2, y, text, ha="center", va="bottom", fontsize=10)

def passes_group_prevalence(feature, biom_table, metadata_df, min_prev=0.10):
    sample_ids = biom_table.ids(axis="sample")
    feature_counts = biom_table.data(feature, axis="observation")
    present = pd.Series(feature_counts > 0, index=sample_ids)

    present = present.loc[present.index.intersection(metadata_df.index)]
    md = metadata_df.loc[present.index]

    for group in ["Healthy", "Acne_NL", "Acne_L"]:
        group_samples = md.index[md["group"] == group]
        if len(group_samples) == 0:
            return False
        if present.loc[group_samples].mean() < min_prev:
            return False

    return True

In [262]:
# Load metadata
metadata_df = pd.read_csv(METADATA_TSV, sep="\t")
# metadata_df = metadata_df.set_index("SampleID")
# metadata_df.index.name = None
metadata_df

,SampleID,c_zone,visual_assessment_in_vivo_number_of_non_inflammatory_lesions_face,zone,sample_type,planned_study_day_of_visit,visual_assessment_in_vivo_number_of_inflammatory_lesions_face,day,subject_randomization_number,visual_assessment_in_vivo_number_of_non_inflammatory_lesions_cheek_right,...,a,cohort,subject_randomization_id,class,subject_ID,subject_ID_CC,zone_CC,group,severity_level,severity_group
0,LAMI.RD308.D16.C1,C1,not applicable,Lesional,skin,Day 16,not applicable,16,308,not applicable,...,33.765638,acne,RD308,acne,PP_308,PP_308C1,Lesional_C1,Acne_L,moderate,moderate Acne_L
1,LAMI.RD310.D21.C1,C1,72,Lesional,skin,Day 21,36,21,310,17,...,31.919478,acne,RD310,acne,PP_310,PP_310C1,Lesional_C1,Acne_L,moderate,moderate Acne_L
2,LAMI.RD305.D21.C3,C3,69,Non-lesional,skin,Day 21,26,21,305,25,...,22.152503,acne,RD305,healthy,PP_305,PP_305C3,Non-lesional_C3,Acne_NL,absent,absent Acne_NL
3,LAMI.RD306.D18.C2,C2,not applicable,Lesional,skin,Day 18,not applicable,18,306,not applicable,...,27.397918,acne,RD306,acne,PP_306,PP_306C2,Lesional_C2,Acne_L,low,low Acne_L
4,LAMI.RD306.D7.C2,C2,90,Lesional,skin,Day 7,13,7,306,23,...,28.819451,acne,RD306,acne,PP_306,PP_306C2,Lesional_C2,Acne_L,moderate,moderate Acne_L
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261,LAMI.RD317.D21.C1,C1,77,Lesional,skin,Day 21,19,21,317,20,...,21.946648,acne,RD317,acne,PP_317,PP_317C1,Lesional_C1,Acne_L,low,low Acne_L
262,LAMI.RD001.D0.C1,C1,not applicable,Non-lesional,skin,Day 0,not applicable,0,1,not applicable,...,26.344240,control,RD001,healthy,PP_1,PP_1C1,Non-lesional_C1,Healthy,absent,absent Healthy
263,LAMI.RD014.D14.C2,C2,not applicable,Non-lesional,skin,Day 14,not applicable,14,14,not applicable,...,16.359699,control,RD014,healthy,PP_14,PP_14C2,Non-lesional_C2,Healthy,absent,absent Healthy
264,LAMI.RD314.D0.C1,C1,55,Lesional,skin,Day 0,31,0,314,16,...,22.494605,acne,RD314,acne,PP_314,PP_314C1,Lesional_C1,Acne_L,low,low Acne_L


In [263]:
# Load taxonomy
taxonomy_df = pd.read_csv(TAXONOMY_TSV, sep="\t")
tax_map = taxonomy_df.set_index("Feature ID")["Taxon"].to_dict()

In [265]:
for region in regions:
    for side in cheek_sides:
        print(f"\nProcessing {region} {side}")

        # Load subject biplot
        biplot_qza = f"{CTF_QZA_DIR}/CTF_subject_biplot_16S_{region}_{side}-cheek.qza"
        ordination = Artifact.load(biplot_qza).view(OrdinationResults)

        feature_coords = ordination.features.iloc[:, :3].copy()
        feature_coords.columns = ["PC1", "PC2", "PC3"]

        # Load BIOM
        if region == "V1-V3":
            biom_path = f"{BIOM_DIR}/179426_feature-table_16S_V1V3_rare-11054_sampleIDfixed.biom"
        elif region == "V4":
            biom_path = f"{BIOM_DIR}/174951_feature-table_16S_V4_rare-3769_sampleIDfixed.biom"

        biom_table = load_table(biom_path)

        prevalence = (
            (biom_table.matrix_data > 0)
            .sum(axis=1)
            .A1 / biom_table.shape[1]
        )

        prevalent_features = biom_table.ids(axis="observation")[prevalence >= 0.15]

        feature_coords = feature_coords.loc[
            feature_coords.index.isin(prevalent_features)
        ]

        feature_coords["Taxon"] = feature_coords.index.map(
            lambda fid: extract_last_taxon(tax_map.get(fid))
        )

        feature_coords["magnitude"] = np.sqrt(
            feature_coords.PC1**2 + feature_coords.PC2**2
        )

        top_features = (
            feature_coords
            .sort_values("magnitude", ascending=False)
            .head(10)
            .index
        )

        # RCLR table
        rclr_qza = rclr_transformation(biom_table)

        rclr_df = pd.DataFrame(
            rclr_qza.matrix_data.toarray(),
            index=rclr_qza.ids(axis="observation"),
            columns=rclr_qza.ids(axis="sample")
        ).fillna(0).T

        rclr_df = (
            rclr_df
            .merge(
                metadata_df[["SampleID", "group"]],
                left_index=True,
                right_on="SampleID",
                how="inner"
            )
            .set_index("SampleID")
        )

        rclr_df["Group"] = pd.Categorical(
            rclr_df["group"],
            categories=GROUP_ORDER,
            ordered=True
        )

        # FIRST PASS: collect all p-values (region × side)
        pairs = [
            ("Healthy", "Acne_NL"),
            ("Healthy", "Acne_L"),
            ("Acne_NL", "Acne_L")
        ]

        pval_records = []

        for feature in top_features:
            plot_df = rclr_df[[feature, "Group"]].dropna()

            for g1, g2 in pairs:
                d1 = plot_df.loc[plot_df.Group == g1, feature]
                d2 = plot_df.loc[plot_df.Group == g2, feature]

                if len(d1) < 2 or len(d2) < 2:
                    continue

                _, p = mannwhitneyu(d1, d2)
                pval_records.append((feature, g1, g2, p))

        pval_df = pd.DataFrame(
            pval_records,
            columns=["feature", "g1", "g2", "p"]
        )

        # Apply BH FDR correction across ALL tests in this region × side
        pval_df["p_adj"] = multipletests(
            pval_df["p"],
            method="fdr_bh"
        )[1]

        # SECOND PASS: plot + annotate using FDR-adjusted p
        for i, feature in enumerate(top_features):

            fig, ax = plt.subplots(figsize=(4, 6))
            plot_df = rclr_df[[feature, "Group"]].dropna()

            sns.boxplot(
                x="Group",
                y=feature,
                data=plot_df,
                palette=group_colors,
                width=0.4,
                dodge=False,
                ax=ax
            )

            sns.stripplot(
                x="Group",
                y=feature,
                data=plot_df,
                palette=group_colors,
                jitter=True,
                size=3,
                linewidth=0.6,
                dodge=False,
                ax=ax
            )

            ax.set_ylabel(
                f"{region} {side} cheek RCLR\n{feature_coords.loc[feature, 'Taxon']} ASV",
                fontsize=11
            )
            ax.set_xlabel("")
            ax.set_xticklabels(["Healthy", "Acne NL", "Acne L"])

            ymax = plot_df[feature].max() * 1.3
            significant_found = False

            for j, (g1, g2) in enumerate(pairs):
                row = pval_df[
                    (pval_df.feature == feature) &
                    (pval_df.g1 == g1) &
                    (pval_df.g2 == g2)
                ]

                if row.empty:
                    continue

                p = row["p_adj"].iloc[0]

                if p <= 0.05:
                    significant_found = True
                    stars = "***" if p < 0.001 else "**" if p < 0.01 else "*"

                    add_significance_bracket(
                        ax,
                        j,
                        j + 1,
                        ymax * (1 + j * 0.12),
                        f"{stars} p={p:.2e}"
                    )

            plt.tight_layout()

            if (
                significant_found
                and feature_coords.loc[feature, "Taxon"].startswith("g__")
            ):
                plt.savefig(
                    f"{OUTDIR}/{region}_{side}_{feature_coords.loc[feature, 'Taxon']}_{i+1}.png",
                    dpi=600
                )

            plt.close()



Processing V1-V3 left


/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/categorical.py:641: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped_vals = vals.groupby(grouper)
/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_2887/942675022.py:122: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(
/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context('mode.use_inf_as_na', True):
/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/_oldcore.py:1119: FutureWarning: u


Processing V1-V3 right


/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_2887/942675022.py:122: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(
/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context('mode.use_inf_as_na', True):
/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context('mode.use_inf_as_na', True):
/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/_oldcore.py:1075: FutureWarning: When grouping with a length-1 list-like, you will need to pass a length-1 tuple t


Processing V4 left


/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_2887/942675022.py:122: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(
/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context('mode.use_inf_as_na', True):
/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context('mode.use_inf_as_na', True):
/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/_oldcore.py:1075: FutureWarning: When grouping with a length-1 list-like, you will need to pass a length-1 tuple t


Processing V4 right


/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/categorical.py:641: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped_vals = vals.groupby(grouper)
/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_2887/942675022.py:122: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(
/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context('mode.use_inf_as_na', True):
/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/_oldcore.py:1119: FutureWarning: u